In [1]:
import os
import pathlib
import sys
import socket

hostname = socket.gethostname()
print(hostname)


abs_path = str(pathlib.Path(hostname).parent.absolute())
WORKDIR = abs_path[:- (len(abs_path.split('/')[-2] + abs_path.split('/')[-1])+1)]



if "mimi" in hostname:
    DATA_DIR = "/mn/vann/franzihe/"

INPUT_DATA_DIR = os.path.join(DATA_DIR, 'input/ceilometer')
OUTPUT_DATA_DIR = os.path.join(DATA_DIR, 'input/ceilometer/cl61_corrected')
UTILS_DIR = os.path.join(WORKDIR, 'utils')

sys.path.append(UTILS_DIR)

# make figure directory to put in reflectivity processed data
# try:
#     new_dir = pathlib.Path(FIG_DIR)
#     new_dir.mkdir(parents=True, exist_ok=True)
# except OSError:
#     pass

# make data directory to put in ceilometer processed data
try:
    new_dir = pathlib.Path(OUTPUT_DATA_DIR)
    new_dir.mkdir(parents=True, exist_ok=True)
except OSError:
    pass

mimi.uio.no


In [2]:
# supress warnings
import warnings
warnings.filterwarnings('ignore') # don't output warnings

# import packages
from imports import(pd, xr, fct, np, glob, plt, date2num, cm, datetime)

In [3]:
# reload imports
%load_ext autoreload
%autoreload 2

In [4]:
current_date = datetime.datetime.utcnow().strftime('%Y-%m-%d')

In [5]:
date_ranges = {
    "2023": pd.date_range("2023-02-23", "2023-03-13").strftime("%Y%m%d").tolist(),
}

In [6]:
ds_cl61 = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR,"cl61_shed/cl61*2023*.nc"))), decode_times=True)

In [7]:
ds_live = xr.open_mfdataset(sorted(glob(os.path.join(INPUT_DATA_DIR,"cl61_shed/live_*2023*.nc"))), decode_times=True)

In [12]:
ds_live.longitude.values[0], ds_live.latitude.values[0]

(16.0337, 69.2955)

In [ ]:
for key, dates in date_ranges.items():
    for date in dates: 
        ds_cl61_clean_time = ds_cl61.sel(time=date)
        ds_cl61_clean_time = fct.copy_all_attrs(ds_live, ds_cl61_clean_time)
        
        # Apply mask: keep only beta_att values greater than noise level
        beta_att_clean = ds_cl61_clean_time['beta_att'].where(ds_cl61_clean_time['beta_att'] > ds_cl61_clean_time['beta_att_noise_level'])
        beta_att_clean.attrs['history'] = f'{current_date}: filter out beta_att values below beta_att_noise_level'
        # replace in dataset
        ds_cl61_clean_time['beta_att'] = beta_att_clean
        # ds_cl61_clean_time['beta_att'].attrs = 
        
        # rename coordinate to match the MetNo coordinate name
        ds_cl61_clean = ds_cl61_clean_time.rename({'range':'altitude'})
        # replace coordinate values after renaming
        altitude = ds_cl61_clean['altitude']+ds_cl61_clean_time['elevation'].isel(time=0)
        ds_cl61_clean = ds_cl61_clean.assign_coords(altitude=altitude.values)
        
        path = os.path.join(OUTPUT_DATA_DIR, f"cl61d_{date}.nc")
        print(f"writing file: ... {path}")
        ds_cl61_clean.to_netcdf(path)

writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230223.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230224.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230225.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230226.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230227.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230228.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230301.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230302.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230303.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230304.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/cl61_corrected/cl61d_20230305.nc
writing file: ... /mn/vann/franzihe/input/ceilometer/c

In [10]:
for var_name, var_data in ds_cl61_clean_time.data_vars.items():
    print(f"Variable: {var_name}", f"  Dimensions: {var_data.dims}", f"  Shape: {var_data.shape}", f"  Data type: {var_data.dtype}", "  Attributes:")
    for attr_key, attr_val in var_data.attrs.items():
        print(f"    {attr_key}: {attr_val}")
    print("-" * 40)

Variable: lon   Dimensions: ('time',)   Shape: (17179,)   Data type: float64   Attributes:
----------------------------------------
Variable: lat   Dimensions: ('time',)   Shape: (17179,)   Data type: float64   Attributes:
----------------------------------------
Variable: elevation   Dimensions: ('time',)   Shape: (17179,)   Data type: float64   Attributes:
    units: m
    long_name: elevation
    standard_name: ground_level_altitude
    comment: measurement site height above or below a fixed reference point, most commonly a reference geoid
----------------------------------------
Variable: beta_att_sum   Dimensions: ('time',)   Shape: (17179,)   Data type: float64   Attributes:
    units: 10^-4sr^-1
    long_name: scaled integral of the attenuated volume backscatter coefficient
----------------------------------------
Variable: beta_att_noise_level   Dimensions: ('time',)   Shape: (17179,)   Data type: float64   Attributes:
    long_name: a unitless number describing the noise level